In [1]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np
import pandas as pd

from src.clustering import load_embeddings, apply_pca, run_kmeans
from src.theme_extraction import extract_keywords_per_cluster
from src.visualization import plot_umap

C:\Users\Arushi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load embeddings
embeddings = load_embeddings("../embeddings/amazon_embeddings.npy")

# Load dataset
df = pd.read_csv("../data/cleaned_amazon_reviews.csv")

print("Embeddings:", embeddings.shape)
print("Dataset:", df.shape)

Loaded embeddings shape: (200000, 384)
Embeddings: (200000, 384)
Dataset: (200000, 13)


In [4]:
assert len(embeddings) == len(df), "Mismatch between embeddings and dataset!"

print("Data aligned")

Data aligned


In [5]:
reduced_embeddings, pca_model = apply_pca(embeddings, n_components=50)

Applying PCA...
Reduced shape: (200000, 50)


In [6]:
n_clusters = 50

labels, kmeans_model = run_kmeans(reduced_embeddings, n_clusters=n_clusters)

df["cluster"] = labels

print("Clustering complete")
df["cluster"].value_counts().head()

Running KMeans with k=50...
Clustering complete


cluster
1     7844
45    6797
26    6768
47    6156
35    5985
Name: count, dtype: int64

In [7]:
cluster_keywords_df = extract_keywords_per_cluster(df, df["cluster"])
cluster_keywords_df.head()

Extracting keywords per cluster...
Processing cluster 0...
Processing cluster 1...
Processing cluster 2...
Processing cluster 3...
Processing cluster 4...
Processing cluster 5...
Processing cluster 6...
Processing cluster 7...
Processing cluster 8...
Processing cluster 9...
Processing cluster 10...
Processing cluster 11...
Processing cluster 12...
Processing cluster 13...
Processing cluster 14...
Processing cluster 15...
Processing cluster 16...
Processing cluster 17...
Processing cluster 18...
Processing cluster 19...
Processing cluster 20...
Processing cluster 21...
Processing cluster 22...
Processing cluster 23...
Processing cluster 24...
Processing cluster 25...
Processing cluster 26...
Processing cluster 27...
Processing cluster 28...
Processing cluster 29...
Processing cluster 30...
Processing cluster 31...
Processing cluster 32...
Processing cluster 33...
Processing cluster 34...
Processing cluster 35...
Processing cluster 36...
Processing cluster 37...
Processing cluster 38...


,cluster,keywords
0,0,"price, amazon, cup, order, flavor, beans, tast..."
1,1,"flavor, taste, cup, strong, bold, roast, bitte..."
2,2,"dogs, loves, treat, training, small, little, e..."
3,3,"taste, sweet, use, free, syrup, flavor, sugar ..."
4,4,"candy, box, gift, chocolate, bought, little, k..."


In [8]:
# Filter out remaining weak words from extracted keywords
weak_words = set(["good", "great", "like", "love", "product", "buy", "just",
                  "really", "nice", "best", "better", "well", "also"])

def clean_keywords(keyword_str):
    return ", ".join([kw for kw in keyword_str.split(", ") if kw not in weak_words])

cluster_keywords_df["keywords"] = cluster_keywords_df["keywords"].apply(clean_keywords)
cluster_keywords_df.head()

,cluster,keywords
0,0,"price, amazon, cup, order, flavor, beans, tast..."
1,1,"flavor, taste, cup, strong, bold, roast, bitte..."
2,2,"dogs, loves, treat, training, small, little, e..."
3,3,"taste, sweet, use, free, syrup, flavor, sugar ..."
4,4,"candy, box, gift, chocolate, bought, little, k..."


In [9]:
for cluster_id in range(5):
    print(f"\n Cluster {cluster_id}")
    
    # Get keywords
    keywords = cluster_keywords_df[cluster_keywords_df["cluster"] == cluster_id]["keywords"].values[0]
    print("Keywords:", keywords)
    
    # Sample reviews
    sample = df[df["cluster"] == cluster_id]["clean_text"].sample(5, random_state=42)
    
    for review in sample:
        print("-", review[:150])


 Cluster 0
Keywords: price, amazon, cup, order, flavor, beans, taste, box, time, starbucks
- this was terrible coffee wolfgang pucks jamaicamecrazy i received a partial refund but felt i should have gotten a full refund as i can t drink the co
- i generally buy my via from starbucks but buying here saved money and the number of trips i will make to starbucks to buy the instant coffee
- i love ordering on line esp from amazon just as i am about to run out of coffee i get a email re special dealsand i simply order and in a day the prod
- nice price fast shipping a keurig machine is not necessary to brew i found the product to be very sweet so i add it to 12 ounces of hot water rather t
- surprisingly good my favorite is the starbucks africa kitamu however if you purchase a two pack of the maxwell french roast from amazon it s a better 

 Cluster 1
Keywords: flavor, taste, cup, strong, bold, roast, bitter, flavored, dark, vanilla
- i ve been an avid coffee drinker for many years but was 

In [10]:
cluster_keywords_df["domain"] = cluster_keywords_df["keywords"].apply(
    lambda x: "pet_food" if any(w in x for w in ["dog","cat","treats"]) else "human_food"
)

In [11]:
plot_umap(reduced_embeddings, df["cluster"], save_path="../analysis/embedding_clusters.png")

Running UMAP for visualization...


C:\Users\Arushi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved plot to ../analysis/embedding_clusters.png


In [12]:
cluster_keywords_df.to_csv("../analysis/cluster_keywords.csv", index=False)
df.to_csv("../analysis/reviews_with_clusters.csv", index=False)